# AML - GAUSS    
This script performs the GAUSS gene set analysis for genes from GO, KEGG, and MSigDB. For MSigDB, gene sets are filtered based on their relevance to acute myeloid leukemia, and only those related to AML are retained for this analysis. Before running the script, set the number of CPUs to 4 to enable parallel processing in GAUSS. The second half of the script generates plots of the final results.

In [ ]:
# Laod packages 
library(GAUSS)
library(data.table)
library(tidyverse)
library(biomaRt)
library(readr)
library(xlsx)
library(paletteer)

# GENERATE GDC SOMATIC MUTATION GENE-SET FILE 

In [ ]:
# Copy somatic gene-sets from workspace bucket 
system("rm -rf gene_list")
system("mkdir -p gene_list")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/gene_sets/somatic_mutations/* gene_list")

In [ ]:
# Format genes with somatic mutations into gmt format 
somatic_mutations <- list()

for (file in list.files(path = "gene_list")) {

    data <- fread(file.path("gene_list", file))

    if (all(is.na(data$symbol))) {
        break
    }

    data <- 
    data.frame(GeneSet = gsub(".tsv","", file), 
               DESC = "GDC",
               Genes = paste(data$symbol, collapse = ","))

    somatic_mutations[[file]] <- data
}
gmt <- do.call(rbind, somatic_mutations)

readr::write_tsv(gmt, "somatic_mutations.gmt")

# GENERATE AML MSIGDB GENE-SET FILE

In [ ]:
# Download and format migdb file 
system("wget https://data.broadinstitute.org/gsea-msigdb/msigdb/release/2025.1.Hs/msigdb.v2025.1.Hs.symbols.gmt", 
       intern=TRUE)
system("grep 'AML' msigdb.v2025.1.Hs.symbols.gmt > filtered_msigdb.v2025.1.Hs.symbols.gmt", 
       intern=TRUE)

In [ ]:
# Filter msigdb gmt file for gene-sets related to acute myeloid leukaemia 
gmt <- fread("filtered_msigdb.v2025.1.Hs.symbols.gmt", fill = Inf, header = FALSE, sep = "\t")

colnames(gmt) <- as.character(paste0("V", 1:ncol(gmt)))
colnames(gmt)[1:2] <- c("GeneSet", "DESC")

gmt <- filter(gmt, grepl("AML", GeneSet))

gmt$Genes <- apply(gmt[, 3:ncol(gmt)], 1, function(x) paste(x, collapse = ","))

gmt[, Genes := gsub(",+$", "", Genes)]

gmt <- dplyr::select(as.data.frame(gmt), GeneSet, DESC, Genes)

# Remove gene which are not directly related to acute myeloid leukaemia 
exclude <- c(
  "MAML1_TARGET_GENES", 
  "KEGG_MEDICUS_VARIANT_AML1_ETO_FUSION_TO_PU.1_MEDIATED_TRANSCRIPTION",
  "KEGG_MEDICUS_VARIANT_TEL_AML1_FUSION_TO_TRANSCRIPTIONAL_REPRESSION", 
  "ONO_AML1_TARGETS_DN", 
  "ONO_AML1_TARGETS_UP"
)

gmt <- filter(gmt, !(GeneSet %in% exclude))

readr::write_tsv(gmt, "aml_msigdb.gmt")

# FORMAT SAIGE-GENE RESULTS

In [ ]:
# Create list for cauchy p-values from SAIGE-GENE+ 
cauchy_data <- list()

# Load results from set-based analysis and annotate with gene symbols 
data <- fread("merged_saige_gene_step02/EUR_aml_phenotype_high_confidence_merged_chr_deprivation_index")
data <- filter(data, Group == "Cauchy")

# Annotate with hgnc gene symbol 
mart <- useEnsembl(biomart = "genes", dataset = "hsapiens_gene_ensembl")

annotations <- getBM(
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    filters = "ensembl_gene_id",
    values = unique(data$Region),
    mart = mart
)

colnames(annotations) <- c("Region", "Symbol")

annotations <- filter(annotations, nzchar(Symbol))

data <- full_join(data, annotations, by ="Region") %>% 
mutate(Symbol = ifelse(is.na(Symbol), Region, Symbol)) %>% 
dplyr::select(Symbol, Pvalue)

# Save to list 
cauchy_data[["EUR-AML-Cauchy-P-Values-Deprivation-Index"]] <- data

In [ ]:
# Load results from set-based analysis and annotate with gene symbols 
data <- fread("merged_saige_gene_step02/EUR_aml_phenotype_high_confidence_merged_chr")
data <- filter(data, Group == "Cauchy")

# Annotate with hgnc gene symbol 
mart <- useEnsembl(biomart = "genes", dataset = "hsapiens_gene_ensembl")

annotations <- getBM(
    attributes = c("ensembl_gene_id", "hgnc_symbol"),
    filters = "ensembl_gene_id",
    values = unique(data$Region),
    mart = mart
)

colnames(annotations) <- c("Region", "Symbol")

annotations <- filter(annotations, nzchar(Symbol))

data <- full_join(data, annotations, by ="Region") %>% 
mutate(Symbol = ifelse(is.na(Symbol), Region, Symbol)) %>% 
dplyr::select(Symbol, Pvalue)

# Save to list 
cauchy_data[["EUR-AML-Cauchy-P-Values"]] <- data

# GAUSS COMMAND 

In [ ]:
# Loop for somatic mutation gene-sets 
for (name in names(cauchy_data)){ 
    data <- cauchy_data[[name]]
    data <- dplyr::select(data, Symbol, Pvalue)
    write_tsv(data, "GAUSS-Gene-Set.txt", col_names=FALSE)
    GAUSS_All(summary_file = "GAUSS-Gene-Set.txt", 
              gene_name = 1, 
              pv_name = 2, 
              output_file = paste0("AML-SOMATIC-", name), 
              gmt = "somatic_mutations.gmt", 
              ags = "prec",
              verbose = TRUE,
              parallel = TRUE, 
              jobs = 4, 
              start = 1,
              stop = 6)
}

In [ ]:
# Loop for aml msigbd gene-sets 
for (name in names(cauchy_data)){ 
    data <- cauchy_data[[name]]
    data <- dplyr::select(data, Symbol, Pvalue)
    GAUSS_All(summary_file = "GAUSS-Gene-Set.txt", 
              gene_name = 1, 
              pv_name = 2, 
              output_file = paste0("AML-MSIGBD-", name), 
              gmt = "aml_msigdb.gmt", 
              ags = "prec",
              verbose = TRUE,
              parallel = TRUE, 
              jobs = 4,
              start = 1,
              stop = nrow(gmt))
}

In [ ]:
# Loop for kegg gene-sets 
for (name in names(cauchy_data)){ 
    data <- cauchy_data[[name]]
    data <- dplyr::select(data, Symbol, Pvalue)
    write_tsv(data, "GAUSS-Gene-Set.txt", col_names=FALSE)
    GAUSS_All(summary_file = "GAUSS-Gene-Set.txt", 
              gene_name = 1, 
              pv_name = 2, 
              output_file = paste0("AML-KEGG-", name), 
              gmt = "~/GAUSS/example_gmt.txt", 
              ags = "prec",
              verbose = TRUE,
              parallel = TRUE, 
              jobs = 4, 
              start = 1,
              stop = 200)
}

In [ ]:
# Loop for go gene-sets 
for (name in names(cauchy_data)){ 
    data <- cauchy_data[[name]]
    data <- dplyr::select(data, Symbol, Pvalue)
    GAUSS_All(summary_file = "GAUSS-Gene-Set.txt", 
          gene_name = 1, 
          pv_name = 2, 
          output_file = paste0("AML-GO-", name), 
          gmt = "~/GAUSS/example_gmt2.txt", 
          ags = "prec",
          verbose = TRUE,
          parallel = TRUE, 
          jobs = 4,
          start = 1,
          stop = 500)
}

In [ ]:
# Save gauss results 
system("gsutil cp AML-*.out gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gauss/gauss_aml_files/", 
       intern=TRUE)

In [ ]:
# Download gauss results 
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gauss/gauss_aml_files/AML-*.out .", 
       intern=TRUE)

# SAVE GAUSS RESULTS  

In [ ]:
# KEGG gene-set
guass_kegg <- fread("AML-KEGG-EUR-AML-Cauchy-P-Values.out")
guass_kegg$Source <- rep("KEGG GENE-SETS", nrow(guass_kegg))

# GO gene-set 
gauss_go <- fread("AML-GO-EUR-AML-Cauchy-P-Values.out")
gauss_go$Source <- rep("GO GENE-SETS", nrow(gauss_go))

# Acute myeloid leukemia gene-set 
gauss_msigdb <- fread("AML-MSIGBD-EUR-AML-Cauchy-P-Values.out")
gauss_msigdb$Source <- rep("AML MSIGBD GENE-SETS", nrow(gauss_msigdb))

# Somatic gene-set
guass_somatic <- fread("AML-SOMATIC-EUR-AML-Cauchy-P-Values.out")
guass_somatic$Source <- rep("SOMATIC GENE-SETS", nrow(guass_somatic))

In [ ]:
# Combine all gene-sets 
gauss_results <- list(guass_kegg, gauss_go, gauss_msigdb, guass_somatic)
gauss_results <- do.call(rbind, gauss_results)
gauss_results <- arrange(gauss_results, pvalue)
write.xlsx(as.data.frame(gauss_results), 
           "GAUSS-EUR-AML-Cauchy-P-Values.xlsx")

In [ ]:
# KEGG gene-set
guass_kegg <- fread("AML-KEGG-EUR-AML-Cauchy-P-Values-Deprivation-Index.out")
guass_kegg$Source <- rep("KEGG GENE-SETS", nrow(guass_kegg))

# GO gene-set 
gauss_go <- fread("AML-GO-EUR-AML-Cauchy-P-Values-Deprivation-Index.out")
gauss_go$Source <- rep("GO GENE-SETS", nrow(gauss_go))

# Acute myeloid leukemia gene-set 
gauss_msigdb <- fread("AML-MSIGBD-EUR-AML-Cauchy-P-Values-Deprivation-Index.out")
gauss_msigdb$Source <- rep("AML MSIGBD GENE-SETS", nrow(gauss_msigdb))

# Somatic gene-set
guass_somatic <- fread("AML-SOMATIC-EUR-AML-Cauchy-P-Values.out")
guass_somatic$Source <- rep("SOMATIC GENE-SETS", nrow(guass_somatic))

In [ ]:
# Combine all gene-sets 
gauss_results <- list(guass_kegg, gauss_go, gauss_msigdb)
gauss_results <- do.call(rbind, gauss_results)
gauss_results <- arrange(gauss_results, pvalue)
write.xlsx(as.data.frame(gauss_results), 
           "GAUSS-EUR-AML-Cauchy-P-Values-Deprivation-Index.xlsx")

In [ ]:
# Check the results  
gauss_results$bonferroni <- p.adjust(gauss_results$pvalue, method = "bonferroni")
head(arrange(gauss_results, pvalue))

# GAUSS PLOT 

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8)

In [ ]:
# Compute bonferroni threshold
threshold <- -log10(0.05 / nrow(gauss_results))

# Compute negative log10 p-values 
gauss_results <- gauss_results %>%
mutate(
    Negative_Log = -log10(pvalue),
    bonferroni = p.adjust(pvalue, method = "bonferroni"),
    Negative_Log_Bonferroni = -log10(bonferroni),
    GeneSet = gsub("_", " ", GeneSet) 
)

# Extract top 2 gene sets per Source (for labeling in facets)
top_sets <- gauss_results %>%
filter(!is.na(GeneSet) & GeneSet != "") %>%
group_by(Source) %>%
arrange(pvalue) %>%
slice(1:2) %>%
ungroup()

# Label for top gene-sets 
top_sets <- top_sets %>%
mutate(GeneSet = ifelse(grepl("MUELLER", GeneSet), GeneSet, str_wrap(GeneSet, width = 32))) %>% 
mutate(GeneSet = gsub("-"," ",GeneSet))
top_sets$nudge_y <- ifelse(grepl("GO", top_sets$GeneSet), 2, 1)

# Create the plot
plot <- gauss_results %>%
filter(!is.na(GeneSet) & GeneSet != "") %>%
ggplot(aes(x = GeneSet, y = Negative_Log)) +
geom_segment(aes(x = GeneSet, xend = GeneSet, y = 0, yend = Negative_Log), 
             size = 0.5, color = "black") +
geom_point(aes(color = Source, size = Negative_Log)) +
geom_hline(yintercept = threshold, linetype = "dashed") +
geom_text(
    data = top_sets,
    aes(x = GeneSet, y = Negative_Log, label = GeneSet),
    size = 3,
    nudge_y = top_sets$nudge_y, 
    show.legend = FALSE
) +
theme_minimal() +
theme(
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank(),
    axis.title.x = element_blank(),
    panel.grid.major.x = element_blank(),
    panel.grid.minor.x = element_blank(), 
    strip.text = element_text(size = 8)
) +

labs(y = "-log10(p-value)") +
paletteer::scale_color_paletteer_d("LaCroixColoR::Lemon") +
facet_wrap(vars(Source), scales = "free_x", ncol = 1) +
guides(color = "none", size = "none")

# Display plot
plot

# Save plot 
ggsave("gauss_aml_cauchy_p_values_deprivation_index_plot.png", 
       plot = plot, 
       width = 15, 
       height = 8, 
       dpi = 300)

# Save plot 
saveRDS(plot, file = 
        paste0("gauss_aml_cauchy_p_values_deprivation_index_plot.rds"))